# 02 — Baseline Models

Baseline-first notebook with leakage-safe seasonal and raw-derived recovery baselines.

In [ ]:
from pathlib import Path
import json
import random
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
RANDOM_SEED = 2026
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

def find_package_paths():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        # Final GitHub layout: repo_root/Raw Data and repo_root/part3_forecasting/notebooks
        if (candidate / "Raw Data").exists() and (candidate / "part3_forecasting" / "notebooks").exists():
            return candidate, candidate / "part3_forecasting"
        # Backward-compatible layout used during development: package_root/Raw Data and package_root/notebooks
        if (candidate / "Raw Data").exists() and (candidate / "notebooks").exists():
            return candidate, candidate
    raise FileNotFoundError("Cannot locate repository root with Raw Data and forecasting notebooks")

REPO_ROOT, PACKAGE_ROOT = find_package_paths()
RAW_DIR = REPO_ROOT / "Raw Data"
NB_DIR = PACKAGE_ROOT / "notebooks"
ARTIFACT_DIR = PACKAGE_ROOT / "artifacts"
REPORT_DIR = PACKAGE_ROOT / "reports"
OUTPUT_DIR = PACKAGE_ROOT / "outputs"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("PACKAGE_ROOT =", PACKAGE_ROOT)
print("RAW_DIR =", RAW_DIR)


## Load Feature Store

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

train = pd.read_parquet(ARTIFACT_DIR / "train_features.parquet")
test = pd.read_parquet(ARTIFACT_DIR / "test_features.parquet")
feature_cols = json.loads((ARTIFACT_DIR / "feature_cols.json").read_text(encoding="utf-8"))
train["Date"] = pd.to_datetime(train["Date"])
test["Date"] = pd.to_datetime(test["Date"])
print(train.shape, test.shape, len(feature_cols))


## Baseline Helpers

In [ ]:
def metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": float(r2_score(y_true, y_pred)),
    }

def derive_recovery_anchor(history, target="Revenue"):
    h = history.copy()
    h["year"] = h["Date"].dt.year
    pre = h[h["year"].between(2016, 2018)][target]
    if pre.empty:
        pre = h[h["year"] <= min(2018, h["year"].max())][target]
    latest_year = int(h["year"].max())
    latest = h[h["year"].eq(latest_year)][target]
    if latest.empty:
        latest = h[target].tail(365)
    pre_mean = float(pre.mean())
    latest_mean = float(latest.mean())
    # Raw-derived compromise between mature pre-Covid demand and latest observed recovery level.
    anchor = 0.5 * pre_mean + 0.5 * latest_mean
    return anchor, {"pre_maturity_mean": pre_mean, "latest_year": latest_year, "latest_mean": latest_mean, "anchor": anchor}

def seasonal_recovery_forecast(history, pred_dates, target="Revenue"):
    h = history[["Date", target]].dropna().copy()
    h["year"] = h["Date"].dt.year
    h["month"] = h["Date"].dt.month
    h["day_of_week"] = h["Date"].dt.dayofweek
    h["month_day"] = h["Date"].dt.strftime("%m-%d")
    shape_source = h[h["year"].between(2013, 2018)]
    if shape_source.empty:
        shape_source = h
    daily_ratio = shape_source.groupby("month_day")[target].mean() / shape_source[target].mean()
    month_ratio = shape_source.groupby("month")[target].mean() / shape_source[target].mean()
    dow_ratio = shape_source.groupby("day_of_week")[target].mean() / shape_source[target].mean()
    anchor, info = derive_recovery_anchor(history, target)
    pred = pd.DataFrame({"Date": pd.to_datetime(pred_dates)})
    pred["month"] = pred["Date"].dt.month
    pred["day_of_week"] = pred["Date"].dt.dayofweek
    pred["month_day"] = pred["Date"].dt.strftime("%m-%d")
    ratio = pred["month_day"].map(daily_ratio).fillna(pred["month"].map(month_ratio)).fillna(1.0)
    ratio = 0.85 * ratio + 0.15 * pred["day_of_week"].map(dow_ratio).fillna(1.0)
    values = ratio.to_numpy(dtype=float)
    values = values / np.nanmean(values) * anchor
    return np.clip(values, 0, None), info

FOLDS = [
    ("Fold_2017", "2016-12-31", "2017-01-01", "2017-12-31"),
    ("Fold_2018", "2017-12-31", "2018-01-01", "2018-12-31"),
    ("Fold_2021", "2020-12-31", "2021-01-01", "2021-12-31"),
    ("Fold_2022", "2021-12-31", "2022-01-01", "2022-12-31"),
    ("Fold_H548_2021_2022", "2021-06-30", "2021-07-01", "2022-12-30"),
]


## Rolling-Origin Baseline Evaluation

In [ ]:
rows = []
oof_parts = []
for fold, train_end, val_start, val_end in FOLDS:
    hist = train[train["Date"] <= train_end].copy()
    val = train[train["Date"].between(val_start, val_end)].copy()
    if len(hist) < 365 or val.empty:
        continue
    for target in ["Revenue", "COGS"]:
        pred, anchor_info = seasonal_recovery_forecast(hist, val["Date"], target)
        m = metrics(val[target], pred)
        rows.append({"fold": fold, "model": "raw_recovery_seasonal", "target": target, **m, **anchor_info})
        oof_parts.append(pd.DataFrame({"Date": val["Date"], "target": target, "actual": val[target], "prediction": pred, "model": "raw_recovery_seasonal", "fold": fold}))

        naive_map = hist.assign(month_day=hist["Date"].dt.strftime("%m-%d")).groupby("month_day")[target].median()
        naive = val["Date"].dt.strftime("%m-%d").map(naive_map).fillna(hist[target].median()).to_numpy()
        rows.append({"fold": fold, "model": "seasonal_naive", "target": target, **metrics(val[target], naive)})
        oof_parts.append(pd.DataFrame({"Date": val["Date"], "target": target, "actual": val[target], "prediction": naive, "model": "seasonal_naive", "fold": fold}))

    ridge_features = [c for c in feature_cols if c in hist.columns]
    for target in ["Revenue", "COGS"]:
        model = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("ridge", Ridge(alpha=20.0))])
        model.fit(hist[ridge_features], np.log1p(hist[target]))
        pred = np.expm1(model.predict(val[ridge_features])).clip(0)
        rows.append({"fold": fold, "model": "ridge_log", "target": target, **metrics(val[target], pred)})
        oof_parts.append(pd.DataFrame({"Date": val["Date"], "target": target, "actual": val[target], "prediction": pred, "model": "ridge_log", "fold": fold}))

baseline_metrics = pd.DataFrame(rows).sort_values(["target", "MAE"])
baseline_oof = pd.concat(oof_parts, ignore_index=True)
baseline_metrics.to_csv(REPORT_DIR / "baseline_metrics.csv", index=False)
baseline_oof.to_parquet(ARTIFACT_DIR / "baseline_oof.parquet", index=False)
baseline_metrics.head(20)


## Baseline Test Predictions

In [ ]:
test_rows = []
full_hist = train[train["Date"].between("2013-01-01", "2022-12-31")].copy()
for target in ["Revenue", "COGS"]:
    pred, anchor_info = seasonal_recovery_forecast(full_hist, test["Date"], target)
    part = pd.DataFrame({"Date": test["Date"], target: pred, "model": "raw_recovery_seasonal"})
    test_rows.append(part)
baseline_test = test_rows[0].merge(test_rows[1], on=["Date", "model"])
baseline_test.to_parquet(ARTIFACT_DIR / "baseline_test_predictions.parquet", index=False)
anchor_report = {
    "Revenue": derive_recovery_anchor(full_hist, "Revenue")[1],
    "COGS": derive_recovery_anchor(full_hist, "COGS")[1],
}
(REPORT_DIR / "02_recovery_anchor_summary.json").write_text(json.dumps(anchor_report, indent=2), encoding="utf-8")
baseline_test[["Revenue", "COGS"]].describe()
